# Predicting Airline Satisfaction — GBDT Blend Baseline (Playground S6E10)

**Metric:** ROC AUC on the probability of `satisfaction`.

**What this notebook does**
1. Loads the data and runs a quick EDA.
2. Builds a few features from the service ratings and delays.
3. Trains LightGBM, XGBoost and CatBoost with the same 5-fold stratified CV.
4. Blends the three models using out-of-fold (OOF) predictions.
5. Writes `submission.csv`, plus OOF/test predictions and an experiment log for later work.

**First run (Oct 10, 2026):** LightGBM 0.95877 · XGBoost 0.95854 · CatBoost 0.95818 · **Blend CV 0.95885**

**Rule of thumb:** trust your CV score more than the public leaderboard. Every change you try should move the CV AUC — log it in the experiment log at the bottom.

**Settings:** use a GPU accelerator (Settings → Accelerator) so XGBoost and CatBoost run fast. Set `QUICK_MODE = True` while experimenting.

In [ ]:
# ---------------- Settings ----------------
QUICK_MODE   = False   # True = subsample train + 3 folds for fast experiments
QUICK_ROWS   = 100_000
N_FOLDS      = 5
SEED         = 42
USE_MODELS   = ["lgb", "xgb", "cat"]   # drop any you don't want
USE_ORIGINAL = False   # True = append the original Airline Passenger Satisfaction data (attach it as an input first)
EXPERIMENT_NOTE = "baseline: raw features + rating/delay aggregates"

In [ ]:
import os, glob, re, shutil, time, itertools, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 60)
HAS_GPU = shutil.which("nvidia-smi") is not None
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def is_cat(s):
    # True for text/category columns on any pandas version
    return not pd.api.types.is_numeric_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype)

print("GPU available:", HAS_GPU, "| output dir:", OUT_DIR)

## 1. Load data

In [ ]:
def find_file(name):
    hits = [p for p in glob.glob(f"/kaggle/input/**/{name}", recursive=True) if "s6e10" in p or "playground" in p]
    hits = hits or glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(f"./**/{name}", recursive=True)
    assert hits, f"Could not find {name}. Add the competition data via 'Add Input'."
    return hits[0]

train = pd.read_csv(find_file("train.csv"))
test  = pd.read_csv(find_file("test.csv"))
sub   = pd.read_csv(find_file("sample_submission.csv"))
TARGET = "satisfaction"
ID = "id"
print("train", train.shape, "| test", test.shape)
train.head()

In [ ]:
# Optional: append the original dataset the competition was generated from.
# Attach "Airline Passenger Satisfaction" (search it under Add Input), then set USE_ORIGINAL = True.
if USE_ORIGINAL:
    orig_paths = [p for p in glob.glob("/kaggle/input/**/*.csv", recursive=True) if "s6e10" not in p and "playground" not in p]
    orig = pd.concat([pd.read_csv(p) for p in orig_paths], ignore_index=True)
    orig = orig.drop(columns=[c for c in orig.columns if c.lower().startswith("unnamed")])
    orig.columns = [c if c in train.columns else next((t for t in train.columns if t.lower() == c.lower()), c) for c in orig.columns]
    orig = orig[[c for c in train.columns if c in orig.columns]]
    orig[ID] = -1
    print("original rows:", len(orig))
    train["is_original"] = 0; orig["is_original"] = 1; test["is_original"] = 0
    train = pd.concat([train, orig], ignore_index=True)

## 2. Quick EDA

In [ ]:
print("Target distribution:")
print(train[TARGET].value_counts(normalize=True).round(4))
print("\nMissing values (train):")
miss = train.isna().sum(); print(miss[miss > 0] if miss.sum() else "none")
print("\nColumn types:")
summary = pd.DataFrame({"dtype": train.dtypes.astype(str), "nunique": train.nunique(), "example": train.iloc[0]})
summary

In [ ]:
# Target encoding to 0/1. Positive class = "satisfied" (the class the submission asks a probability for).
y_raw = train[TARGET]
if is_cat(y_raw):
    classes = sorted(y_raw.astype(str).unique())
    pos = [c for c in classes if "satisf" in c.lower() and "dis" not in c.lower()]
    pos = pos[0] if pos else classes[-1]
    y = (y_raw.astype(str) == pos).astype(int).values
    print(f"String target -> positive class = '{pos}'")
else:
    y = y_raw.astype(int).values
print("positive rate:", y.mean().round(4))

In [ ]:
# Which raw features separate the classes best? (single-feature AUC, quick signal check)
signal = {}
for c in train.columns.drop([TARGET, ID], errors="ignore"):
    s = train[c]
    if is_cat(s):
        s = s.astype(str).map(pd.Series(y, index=train.index).groupby(s.astype(str)).mean())
    s = s.fillna(s.median())
    if s.nunique() > 1:
        a = roc_auc_score(y, s); signal[c] = max(a, 1 - a)
pd.Series(signal).sort_values(ascending=False).round(4).to_frame("single_feature_auc")

## 3. Feature engineering

In [ ]:
def clean_name(c):
    return re.sub(r"[^0-9a-zA-Z_]+", "_", c).strip("_")

full = pd.concat([train.drop(columns=[TARGET]), test], ignore_index=True)
full.columns = [clean_name(c) for c in full.columns]
ID_C = clean_name(ID)

cat_cols = [c for c in full.columns if is_cat(full[c])]
num_cols = [c for c in full.columns if c not in cat_cols + [ID_C, "is_original"]]
rating_cols = [c for c in num_cols
               if full[c].dropna().between(0, 5).all() and full[c].nunique() <= 6]
delay_cols = [c for c in num_cols if "delay" in c.lower()]
dist_cols  = [c for c in num_cols if "distance" in c.lower()]
print("categorical:", cat_cols)
print("ratings:", rating_cols)
print("delays:", delay_cols, "| distance:", dist_cols)

# Service-rating aggregates (0 usually means "not applicable" in the original data)
if rating_cols:
    R = full[rating_cols]
    full["rating_mean"]  = R.replace(0, np.nan).mean(axis=1)
    full["rating_sum"]   = R.sum(axis=1)
    full["rating_std"]   = R.std(axis=1)
    full["rating_min"]   = R.replace(0, np.nan).min(axis=1)
    full["n_zero"]       = (R == 0).sum(axis=1)
    full["n_high"]       = (R >= 4).sum(axis=1)
    full["n_low"]        = ((R >= 1) & (R <= 2)).sum(axis=1)

# Delay features
if delay_cols:
    D = full[delay_cols]
    if len(delay_cols) == 2:
        a, b = sorted(delay_cols, key=lambda c: "arriv" not in c.lower())  # a = arrival, b = departure
        full[a] = full[a].fillna(full[b])
        full["delay_diff"] = full[a] - full[b]
    full["delay_total"] = full[delay_cols].sum(axis=1)
    full["delay_max"]   = full[delay_cols].max(axis=1)
    full["any_delay"]   = (full["delay_max"] > 0).astype(int)

for c in dist_cols:
    full[f"log_{c}"] = np.log1p(full[c].clip(lower=0))

for c in cat_cols:
    full[c] = full[c].fillna("NA").astype(str).astype("category")

FEATURES = [c for c in full.columns if c != ID_C]
X      = full.iloc[:len(train)][FEATURES].reset_index(drop=True)
X_test = full.iloc[len(train):][FEATURES].reset_index(drop=True)
print("features:", len(FEATURES))

In [ ]:
if QUICK_MODE:
    rng = np.random.RandomState(SEED)
    idx = rng.choice(len(X), size=min(QUICK_ROWS, len(X)), replace=False)
    X, y = X.iloc[idx].reset_index(drop=True), y[idx]
    N_FOLDS = 3
    print("QUICK_MODE: using", len(X), "rows,", N_FOLDS, "folds")

# Original rows are used for training only, never for validation scoring
is_orig = X["is_original"].values == 1 if "is_original" in X.columns else np.zeros(len(X), bool)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
folds = list(skf.split(X, y))

## 4. Models (same folds for every model)

In [ ]:
def fit_lgb(Xtr, ytr, Xva, yva):
    import lightgbm as lgb
    m = lgb.LGBMClassifier(
        n_estimators=5000, learning_rate=0.03, num_leaves=63, min_child_samples=50,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=1.0,
        random_state=SEED, verbose=-1, n_jobs=-1)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric="auc",
          callbacks=[lgb.early_stopping(200, verbose=False)])
    return m

def fit_xgb(Xtr, ytr, Xva, yva):
    import xgboost as xgb
    m = xgb.XGBClassifier(
        n_estimators=5000, learning_rate=0.03, max_depth=7, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.7, reg_lambda=1.0,
        tree_method="hist", device="cuda" if HAS_GPU else "cpu",
        enable_categorical=True, max_cat_to_onehot=8,
        eval_metric="auc", early_stopping_rounds=200, random_state=SEED)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return m

def fit_cat(Xtr, ytr, Xva, yva):
    from catboost import CatBoostClassifier
    cats = [c for c in Xtr.columns if isinstance(Xtr[c].dtype, pd.CategoricalDtype) or not pd.api.types.is_numeric_dtype(Xtr[c])]
    m = CatBoostClassifier(
        iterations=5000, learning_rate=0.06, depth=7, l2_leaf_reg=3,
        eval_metric="Logloss" if HAS_GPU else "AUC",
        task_type="GPU" if HAS_GPU else "CPU",
        od_type="Iter", od_wait=200, random_seed=SEED, verbose=0)
    m.fit(Xtr, ytr, cat_features=cats, eval_set=(Xva, yva), use_best_model=True)
    return m

FITTERS = {"lgb": fit_lgb, "xgb": fit_xgb, "cat": fit_cat}

def prep(df, model):
    # CatBoost wants plain strings for categoricals
    if model == "cat":
        df = df.copy()
        for c in df.columns:
            if isinstance(df[c].dtype, pd.CategoricalDtype):
                df[c] = df[c].astype(str).astype(object)
    return df

def run_cv(model):
    oof = np.zeros(len(X)); pred = np.zeros(len(X_test)); t0 = time.time()
    Xm, Xtm = prep(X, model), prep(X_test, model)
    for k, (tr, va) in enumerate(folds):
        va = va[~is_orig[va]]                       # never score on original rows
        m = FITTERS[model](Xm.iloc[tr], y[tr], Xm.iloc[va], y[va])
        oof[va] = m.predict_proba(Xm.iloc[va])[:, 1]
        pred += m.predict_proba(Xtm)[:, 1] / len(folds)
        print(f"  {model} fold {k}: AUC {roc_auc_score(y[va], oof[va]):.5f}")
    mask = ~is_orig
    auc = roc_auc_score(y[mask], oof[mask])
    print(f"{model} CV AUC: {auc:.5f}  ({time.time()-t0:.0f}s)")
    return oof, pred, auc

results = {}
for name in USE_MODELS:
    results[name] = run_cv(name)

## 5. Blend (weights chosen on OOF, rank-averaged)

In [ ]:
mask = ~is_orig
names = list(results)
oof_r  = {n: rankdata(results[n][0][mask]) / mask.sum() for n in names}
test_r = {n: rankdata(results[n][1]) / len(X_test) for n in names}

best = (-1, None)
grid = np.round(np.arange(0, 1.0001, 0.05), 2)
for w in itertools.product(grid, repeat=len(names)):
    if abs(sum(w) - 1) > 1e-9: continue
    auc = roc_auc_score(y[mask], sum(wi * oof_r[n] for wi, n in zip(w, names)))
    if auc > best[0]: best = (auc, w)

blend_auc, weights = best
print("Single models:", {n: round(float(results[n][2]), 5) for n in names})
print("Blend weights:", {n: float(w) for n, w in zip(names, weights)}, "| blend CV AUC:", round(blend_auc, 5))
blend_test = sum(wi * test_r[n] for wi, n in zip(weights, names))

## 6. Save submission, predictions and experiment log

In [ ]:
sub[TARGET] = blend_test
sub.to_csv(f"{OUT_DIR}/submission.csv", index=False)

# Keep OOF/test predictions so future experiments can be blended with this one
for n in names:
    pd.DataFrame({"oof": results[n][0]}).to_csv(f"{OUT_DIR}/oof_{n}.csv", index=False)
    pd.DataFrame({ID: sub[ID], "pred": results[n][1]}).to_csv(f"{OUT_DIR}/test_{n}.csv", index=False)

log_path = f"{OUT_DIR}/experiments.csv"
row = {"time": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), "note": EXPERIMENT_NOTE,
       "quick_mode": QUICK_MODE, "n_features": len(FEATURES),
       **{f"cv_{n}": round(float(results[n][2]), 5) for n in names}, "cv_blend": round(float(blend_auc), 5),
       "public_lb": ""}
log = pd.concat([pd.read_csv(log_path), pd.DataFrame([row])]) if os.path.exists(log_path) else pd.DataFrame([row])
log.to_csv(log_path, index=False)
print(sub.head()); print(); print(log.tail())

## Next experiments (one at a time — log CV after each)
1. **Original data:** attach the original Airline Passenger Satisfaction dataset and set `USE_ORIGINAL = True`.
2. **Treat ratings as categories:** convert each rating column to `category` and compare CV.
3. **Pairwise interactions:** e.g. `Class × Type_of_Travel`, `Customer_Type × Class` as combined categoricals.
4. **Tuning:** run Optuna on LightGBM in `QUICK_MODE`, then confirm on full data.
5. **More diverse models:** a logistic regression on one-hot features, or a small neural net, often adds to a tree blend.

After each submission, write the public LB score into `experiments.csv` next to its CV score — the gap tells you whether to trust CV.